## 模块：函数 —— 构建可复用的代码块

**驱动问题**：在开发大模型应用时，很多功能会被反复使用，比如调用模型、处理文本、验证权限。我们不能每次都把代码重写一遍。函数就是用来封装这些可复用逻辑的核心工具。本节从基础语法出发，重点掌握能让代码更简洁、更强大的进阶特性：闭包和装饰器。

---

### 1. 函数基础与参数

#### 1.1 定义与调用
函数用 `def` 关键字定义，用 `return` 返回结果。

<img src="./images/51.jpg" alt="1" style="zoom:80%;" />

In [ ]:
# 定义函数
def call_llm(prompt, model="gpt-3.5"):
    """模拟调用大模型"""
    return f"模型 {model} 收到了: {prompt}"

# 使用函数，调用函数，语法：接收返回值 = 函数名(传递参数)
reply = call_llm("你好", model="gpt-4")
print(reply)

reply = call_llm("你是谁？", model="deepseek-V4")
print(reply)

学完了函数定义与调用，发现其实我们在前面已经使用了很多的函数方法了：
- max() min() abs() sqrt()...
- len() range() enumerate() ....
- list() tuple() dict() set() str() int() float() ...


#### 1.2 灵活的 `*args` 和 `**kwargs`

Python 函数参数设计极其灵活，组合运用可应对各种调用场景。

| 参数类型       | 语法示例         | 说明                                                 |
| -------------- | ---------------- | ---------------------------------------------------- |
| 位置参数       | `func(a, b)`     | 按位置一一对应，必须传递                             |
| 关键字参数     | `func(b=2, a=1)` | 指定参数名传递，可不按顺序，可读性强                 |
| 默认参数       | `func(a=10)`     | 调用时可省略，使用默认值。**默认值必须是不可变对象** |
| 可变位置参数   | `*args`          | 接收多余的位置参数，打包为**元组**                   |
| 可变关键字参数 | `**kwargs`       | 接收多余的关键字参数，打包为**字典**                 |

其中*args和**kwagrs这两个特殊参数可以让你编写能接受任意数量参数的函数，在写通用工具时非常有用。

In [ ]:
def user_profile(name, age=18, *hobbies, **extras):
    """展示用户信息"""
    print(f"姓名: {name}, 年龄: {age}")
    if hobbies:
        print(f"爱好: {', '.join(hobbies)}")
    if extras:
        print(f"附加信息: {extras}")

# 各种调用方式
user_profile("Alice", 25, "阅读", "编程", city="北京", job="工程师")
user_profile("白泽")

In [ ]:
def build_payload(model, *messages, **options):
    """构建一个API请求的负载数据"""
    payload = {"model": model, "messages": list(messages)}
    # 把所有额外选项（如 temperature, top_p）合并进去
    payload.update(options)
    return payload

# 调用：可以传入任意数量的消息和任意关键字参数
p = build_payload(
    "gpt-5",
    {"role": "user", "content": "你好"},
    {"role": "assistant", "content": "你好！"},
    temperature=0.7,
    max_tokens=200
)
print(p)
# {'model': 'gpt-5', 'messages': [...]', 'temperature': 0.7, 'max_tokens': 200}

---

### 2. 变量的作用域

理解一个变量在哪里能被访问、在哪里能被修改，是避免 bug 的关键。

-   **局部变量**：在函数内部定义，函数外部无法访问。
-   **全局变量**：在函数外部定义，函数内部只能读取；如果想**修改**它，需要用 `global` 关键字声明。

In [ ]:
# 函数外 定义变量
count = 0

def fun():
    # 函数内定义参数
    # print(count)
    count += 1  # count = count + 1

fun()
print(count)

In [ ]:
request_count = 0  # 全局变量

def track_request():
    global request_count  # 声明要修改全局变量
    request_count += 1
    print(f"API 调用次数: {request_count}")

track_request()  # 输出: API 调用次数: 1
track_request()  # 输出: API 调用次数: 2

In [ ]:
count = 0   # 全局变量
def fun():
    count = 99  # 局部变量
    count += 1
    print(count)

fun()   # 100
fun()   # 100
# 局部变量与全局变量重名，覆盖全局变量；并且局部变量调用之后就被销毁掉了

---

### 3. 闭包 —— 带着“记忆”的函数

闭包是一个函数，但它“记得”自己定义时外部环境的变量。这让你可以创建出带“状态”的函数。

In [ ]:
def create_counter():   # 外函数
    count = 0  # 这个变量会被内层函数“记住”
    def increment():
        nonlocal count  # 声明要修改外层变量，而非创建新的局部变量
        count += 1
        return count
    return increment

# 创建一个独立的计数器
my_counter = create_counter()
print(my_counter())  # 1
print(my_counter())  # 2
print(my_counter())  # 3
# 每一次调用，都修改并“记住”了 count 的最新值

# 与上面的使用global创建的单例计数器不一样，这里可以再创建实例
baize_counter = create_counter()
print(baize_counter())
print(baize_counter())

In [ ]:
def make_multiplier(factor):
    def multiply(number):
        # 闭包：内函数 必须使用 外函数的变量
        return number * factor
    return multiply

# 创建两个不同的实例
my = make_multiplier(2)
print(my(5))

baize = make_multiplier(3)
print(baize(7))

---

### 4. 装饰器 —— 增强函数功能的“语法糖”

装饰器是 Python 中非常强大的特性，它可以让你在**不修改原函数代码**的情况下，给它增加额外的功能（如计时、日志、权限校验）。在大模型应用中，它常用于记录API调用耗时、缓存结果、重试等场景。

#### 4.1 通用装饰器模板

这是一个标准的装饰器写法，你可以在任何需要增强函数的地方套用。

In [ ]:
import time
import functools

def timer(func):
    """一个计时装饰器，可以装饰在任何函数上"""
    @functools.wraps(func)  # 保留原函数的函数名、文档等元信息
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)  # 执行原始函数
        cost = time.perf_counter() - start
        print(f"[计时] {func.__name__} 耗时: {cost:.4f} 秒")
        return result
    return wrapper

# 使用装饰器
@timer
def process_data():
    time.sleep(1.5)     # 模拟函数的执行耗时
    return "处理完成"

# 调用时自动计时
process_data()

# 使用装饰器
@timer
def baize():
    print("我是白泽，正在测试装饰器")
    print("看一下 计时器装饰器 的使用")

baize()

#### 4.2 带参数的装饰器

如果装饰器本身需要参数（比如重试次数），就需要再多套一层函数。

In [ ]:
def retry(max_attempts=3):
    """一个带参数的重试装饰器"""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"重试 {attempt+1}/{max_attempts} 失败: {e}")
            raise Exception(f"函数 {func.__name__} 重试 {max_attempts} 次后仍失败")
        return wrapper
    return decorator

# 使用带参数的装饰器
import random
@retry(max_attempts=5)
def unstable_network_call():
    if random.random() < 0.7:
        raise ConnectionError("网络抖动")
    return "数据获取成功"

print(unstable_network_call())

#### 4.3 实战：缓存装饰器

利用装饰器实现一个简单的缓存功能，避免重复计算，这在处理一些耗时运算时非常有用。

In [ ]:
def cache(func):
    _cache = {}  # 用来存结果
    @functools.wraps(func)
    def wrapper(*args):
        if args not in _cache:
            _cache[args] = func(*args)  # 第一次算，存起来
        return _cache[args]             # 之后直接取
    return wrapper

@cache
def expensive_computation(n):
    print(f"正在进行耗时为 {n} 秒的计算...")
    time.sleep(n)
    return n * 2

print(expensive_computation(2))  # 会等待
print(expensive_computation(3))  # 会等待
print(expensive_computation(2))  # 瞬间返回，因为命中了缓存
print(expensive_computation(3))  # 瞬间返回，因为命中了缓存